# Box's Evolutionary Algorithm

In [5]:
import numpy as np 
from itertools import product


def box_evolutionary(f,x0,delta,epsilon):
    x0=np.array(x0, dtype=float)
    delta =np.array(delta,dtype=float)
    n = len(x0)

    x_bar=x0.copy()
    current_best=x0.copy()
    iteration = 0

    while np.linalg.norm(delta) > epsilon:
        iteration += 1

        candidates=[x_bar.copy()]
        for signs in product([-1,1],repeat=n):
            offset=np.array(signs)*(delta/2)
            candidates.append(x_bar+offset)

        values= [(x,f(x)) for x in candidates]
        x_min,f_min=min(values,key=lambda item:item[1])
        x_bar= x_min.copy()
        
        if np.allclose(x_bar,current_best,atol=1e-8):
            delta /= 2
        else:
            current_best=x_bar.copy()

    return current_best, f(current_best)

In [6]:
def himmelblau(x):
    x1, x2 = x
    return (x1**2 + x2 - 11)**2 + (x1 + x2**2 - 7)**2

x0=[1,1]
delta=[1,1]
epsilon = 0.0001

x_opt,f_opt = box_evolutionary(himmelblau,x0,delta,epsilon)
print(f"Optimal point: {x_opt}, Optimal value: {f_opt}")

Optimal point: [3. 2.], Optimal value: 0.0


# Hooke Jeeves Method

In [9]:
import numpy as np
def hooke_jeeve(func,x0,delta_init,alpha=2.0,epsilon=1e-3,max_iterations=1000):
    def exploaratory_move(x_base,delta):
        x=x_base.copy()

        for i in range(len(x)):
            f=func(x)

            x_plus=x.copy()
            x_plus[i] += delta[i]
            f_plus=func(x_plus)

            x_minus=x.copy()
            x_minus[i] -= delta[i]
            f_minus=func(x_minus)

            f_min=min(f,f_plus,f_minus)
            if f_min==f_plus:
                x=x_plus
            elif f_min==f_minus:
                x=x_minus

        return x, func(x)
    

    x_prev= np.array(x0, dtype=float)
    delta=np.array(delta_init, dtype=float)

    x_curr,f_curr =exploaratory_move(x_prev,delta)

    iterations=0
    while np.linalg.norm(delta) > epsilon and iterations < max_iterations:
        if np.allclose(x_curr,x_prev,atol=1e-8):
            delta /= alpha
        else:
            x_pattern=x_curr + (x_curr - x_prev)
            x_prev = x_curr.copy()

            x_new,f_new=exploaratory_move(x_pattern,delta)  
            if f_new>=f_curr:
                x_curr=x_prev
                delta/=alpha
            else:
                x_curr=x_new
                f_curr=f_new

        iterations += 1

    return x_curr, f_curr, iterations



    

In [10]:
def himmelblau(x):
    x1,x2=x
    return (x1**2 + x2 - 11)**2 + (x1 + x2**2 - 7)**2

x0=[0,0]
delta=[0.5,0.5]
alpha=2
epsilon=1e-3

x_opt,f_opt,iter=hooke_jeeve(himmelblau,x0,delta,alpha,epsilon)

print(f"Optimal point: {x_opt}, Optimal value: {f_opt}, Iterations: {iter}")

Optimal point: [3. 2.], Optimal value: 0.0, Iterations: 12
